# HYPERVIEW2 Compression-Aware Downstream Verification

Notebook sluzy do drugiego etapu downstream: porownania regresji na oryginalach i rekonstrukcjach z modeli kompresji. Zakladany workflow:

1. Klonujemy repo i instalujemy kod.
2. Pobieramy HYPERVIEW2 przez EOTDL.
3. Wskazujemy checkpointy kompresorow albo gotowe katalogi rekonstrukcji.
4. Generujemy rekonstrukcje w canonical layoucie HYPERVIEW2.
5. Porownujemy regresory w trybach:
   - `original_train_to_original_val`,
   - `original_train_to_recon_val`,
   - `recon_train_to_recon_val`.

Metryka glowna to `hyperview_score`, czyli MSE per target znormalizowany przez MSE predykcji sredniej z train setu.

## 1. Repo i zaleznosci

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = 'https://github.com/mhx1467/master-thesis-code.git'
REPO_DIR = Path('/content/hsi')
REPO_REF = 'main'

if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(['git', 'fetch', 'origin'], cwd=REPO_DIR, check=True)
subprocess.run(['git', 'checkout', REPO_REF], cwd=REPO_DIR, check=True)
subprocess.run(['git', 'pull', '--ff-only'], cwd=REPO_DIR, check=True)

os.chdir(REPO_DIR)
print('Repo:', Path.cwd())
print(subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip())

In [ ]:
%pip -q install -e ".[downstream]" eotdl tqdm pandas numpy matplotlib joblib

# This notebook is intended to generate reconstructions from Mamba checkpoints in Colab.
# If this cell fails, switch Colab runtime to GPU and rerun from a clean runtime.
INSTALL_MAMBA = True
INSTALL_OPTIONAL_BOOSTING = False

import subprocess
import sys

import torch

print('Python:', sys.version)
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('Torch CUDA:', torch.version.cuda)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('Mamba checkpoint reconstruction requires a GPU Colab runtime. Select Runtime > Change runtime type > GPU.')

mamba_cmds = [
    [sys.executable, '-m', 'pip', 'install', '--no-build-isolation', 'causal-conv1d>=1.4.0'],
    [sys.executable, '-m', 'pip', 'install', '--no-build-isolation', 'mamba-ssm>=2.3.1'],
]
if INSTALL_MAMBA:
    for cmd in mamba_cmds:
        print('Running:', ' '.join(cmd))
        subprocess.run(cmd, check=True)

try:
    from mamba_ssm import Mamba
    print('mamba-ssm import: ok')
except Exception as exc:
    raise RuntimeError(
        'mamba-ssm is required to load and adapt/generate reconstructions from Mamba checkpoints in this notebook. '
        'Use a fresh GPU Colab runtime, rerun the install cell, or generate reconstructions on the remote GPU.'
    ) from exc

if INSTALL_OPTIONAL_BOOSTING:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'lightgbm', 'catboost', 'xgboost'], check=True)

## 1a. Google Drive cache

Ta sekcja korzysta z plikow przygotowanych na Twoim Google Drive:

- `MyDrive/hsi/checkpoints/mamba_hyperview2_checkpoints_20260525.tar.gz`
- `MyDrive/hsi/data/hyperview2/HYPERVIEW2_20260525.tar.gz`

Checkpointy sa rozpakowywane do `MyDrive/hsi/checkpoints`, dataset do `MyDrive/hsi/data/hyperview2/HYPERVIEW2`, a rekonstrukcje i wyniki trafiaja do `MyDrive/hsi/reconstructions` oraz `MyDrive/hsi/downstream_results`. Pobieranie checkpointow po SSH zostaje jako opcjonalny fallback, ale domyslnie jest wylaczone.


In [ ]:
from pathlib import Path
import os
import subprocess

from google.colab import drive, userdata

drive.mount('/content/drive')

DRIVE_HSI = Path('/content/drive/MyDrive/hsi')
DRIVE_DATA_PARENT = DRIVE_HSI / 'data/hyperview2'
DRIVE_HV2_ROOT = DRIVE_DATA_PARENT / 'HYPERVIEW2'
DRIVE_CHECKPOINTS = DRIVE_HSI / 'checkpoints'
DRIVE_RECONS = DRIVE_HSI / 'reconstructions'
DRIVE_RESULTS = DRIVE_HSI / 'downstream_results'

for path in [DRIVE_DATA_PARENT, DRIVE_CHECKPOINTS, DRIVE_RECONS, DRIVE_RESULTS]:
    path.mkdir(parents=True, exist_ok=True)

REMOTE_HOST = '206.168.83.201'
REMOTE_USER = 'user'
REMOTE_CHECKPOINT_ARCHIVE = '/workspace/hsi/artifacts/export/mamba_hyperview2_checkpoints_20260525.tar.gz'
DRIVE_CHECKPOINT_ARCHIVE = DRIVE_CHECKPOINTS / 'mamba_hyperview2_checkpoints_20260525.tar.gz'
DRIVE_DATA_ARCHIVE = DRIVE_DATA_PARENT / 'HYPERVIEW2_20260525.tar.gz'
DOWNLOAD_REMOTE_CHECKPOINTS_TO_DRIVE = False

if DRIVE_CHECKPOINT_ARCHIVE.exists():
    print('Using checkpoint archive from Drive:', DRIVE_CHECKPOINT_ARCHIVE)
elif DOWNLOAD_REMOTE_CHECKPOINTS_TO_DRIVE:
    ssh_key = userdata.get('REMOTE_SSH_PRIVATE_KEY')
    if not ssh_key:
        raise RuntimeError(
            'Add your remote SSH private key to Colab Secrets as REMOTE_SSH_PRIVATE_KEY, '
            'or manually upload the checkpoint archive to DRIVE_CHECKPOINTS.'
        )
    key_path = Path('/root/.ssh/id_gpu_colab')
    key_path.parent.mkdir(parents=True, exist_ok=True)
    key_path.write_text(ssh_key.strip() + '\n', encoding='utf-8')
    key_path.chmod(0o600)
    subprocess.run([
        'scp', '-o', 'StrictHostKeyChecking=no', '-i', str(key_path),
        f'{REMOTE_USER}@{REMOTE_HOST}:{REMOTE_CHECKPOINT_ARCHIVE}',
        str(DRIVE_CHECKPOINT_ARCHIVE),
    ], check=True)
else:
    print('Checkpoint archive is not present on Drive and remote download is disabled:', DRIVE_CHECKPOINT_ARCHIVE)

if DRIVE_CHECKPOINT_ARCHIVE.exists():
    subprocess.run(['tar', '-xzf', str(DRIVE_CHECKPOINT_ARCHIVE), '-C', str(DRIVE_CHECKPOINTS)], check=True)
    print('Checkpoint files in Drive:')
    for path in sorted(DRIVE_CHECKPOINTS.glob('*.pt')):
        print(' ', path.name, f'{path.stat().st_size / (1024 * 1024):.1f} MiB')


## 2. HYPERVIEW2 z Drive lub EOTDL


In [ ]:
LOCAL_DATA_PARENT = Path('/content/data/hyperview2')
USE_DRIVE_DATA = (
    'DRIVE_DATA_PARENT' in globals()
    and (DRIVE_HV2_ROOT.exists() or DRIVE_DATA_ARCHIVE.exists())
)
DATA_PARENT = DRIVE_DATA_PARENT if USE_DRIVE_DATA else LOCAL_DATA_PARENT
EXPECTED_HV2_ROOT = DATA_PARENT / 'HYPERVIEW2'
DATA_PARENT.mkdir(parents=True, exist_ok=True)

if 'DRIVE_DATA_ARCHIVE' in globals() and DRIVE_DATA_ARCHIVE.exists() and not DRIVE_HV2_ROOT.exists():
    print('Extracting HYPERVIEW2 archive from Drive:', DRIVE_DATA_ARCHIVE)
    subprocess.run(['tar', '-xzf', str(DRIVE_DATA_ARCHIVE), '-C', str(DRIVE_DATA_PARENT)], check=True)


def is_hyperview2_root(path: Path) -> bool:
    return (
        (path / 'train_gt.csv').is_file()
        and (path / 'submission.csv').is_file()
        and (path / 'train' / 'hsi_satellite').is_dir()
        and (path / 'test' / 'hsi_satellite').is_dir()
    )


def find_hyperview2_root(search_root: Path) -> Path | None:
    candidates = [EXPECTED_HV2_ROOT, search_root]
    if search_root.exists():
        candidates.extend(path.parent for path in search_root.rglob('train_gt.csv'))
    for candidate in dict.fromkeys(candidates):
        if is_hyperview2_root(candidate):
            return candidate
    return None


HV2_ROOT = find_hyperview2_root(DATA_PARENT) or EXPECTED_HV2_ROOT
print('Dataset parent:', DATA_PARENT)
print('Dataset root:', HV2_ROOT)
print('Drive archive:', DRIVE_DATA_ARCHIVE if 'DRIVE_DATA_ARCHIVE' in globals() else None)
print('Root ready:', is_hyperview2_root(HV2_ROOT))


In [ ]:
# Uruchom tylko jezeli Colab nie jest jeszcze zalogowany do EOTDL.
# Po logowaniu wykonaj kolejna komorke pobierania.
!eotdl auth login

In [ ]:
FORCE_EOTDL_DOWNLOAD = False

HV2_ROOT = find_hyperview2_root(DATA_PARENT) or EXPECTED_HV2_ROOT
needs_download = not is_hyperview2_root(HV2_ROOT)
partial_expected_dir = EXPECTED_HV2_ROOT.exists() and not is_hyperview2_root(EXPECTED_HV2_ROOT)

if needs_download or FORCE_EOTDL_DOWNLOAD:
    env = os.environ.copy()
    env['EOTDL_STAGE_WORKERS'] = '16'
    cmd = [
        'eotdl', 'datasets', 'get', 'HYPERVIEW2',
        '--version', '2', '--assets', '--path', str(DATA_PARENT), '--verbose',
    ]
    if FORCE_EOTDL_DOWNLOAD or partial_expected_dir:
        cmd.append('--force')
    print('Running:', ' '.join(cmd))
    subprocess.run(cmd, check=True, env=env)
    HV2_ROOT = find_hyperview2_root(DATA_PARENT) or EXPECTED_HV2_ROOT
else:
    print('HYPERVIEW2 already exists, skipping download.')

print('Dataset root:', HV2_ROOT)
print('Root ready:', is_hyperview2_root(HV2_ROOT))

In [ ]:
HV2_ROOT = find_hyperview2_root(DATA_PARENT) or HV2_ROOT
if not is_hyperview2_root(HV2_ROOT):
    print('Current DATA_PARENT tree:')
    for path in sorted(DATA_PARENT.rglob('*'))[:80]:
        print(' ', path.relative_to(DATA_PARENT))
    raise FileNotFoundError('HYPERVIEW2 root is incomplete after download.')

for rel in ['train/hsi_satellite', 'train/hsi_airborne', 'train/msi_satellite', 'test/hsi_satellite', 'test/msi_satellite']:
    directory = HV2_ROOT / rel
    count = len(list(directory.glob('*.npz'))) if directory.exists() else 0
    print(f'{rel:24s} {count:5d} npz files')

## 2a. Cache datasetu na Google Drive

Jesli dataset byl pobrany fallbackiem EOTDL do lokalnego katalogu Colaba, ta komorka kopiuje go na Drive. Gdy archiwum Drive zostalo juz rozpakowane, komorka tylko potwierdza uzywana sciezke.


In [ ]:
import shutil

CACHE_DATASET_ON_DRIVE = True

if CACHE_DATASET_ON_DRIVE:
    if 'DRIVE_HV2_ROOT' not in globals():
        print('Drive cache not configured; continuing with local HV2_ROOT:', HV2_ROOT)
    elif DRIVE_HV2_ROOT.exists() and is_hyperview2_root(DRIVE_HV2_ROOT):
        HV2_ROOT = DRIVE_HV2_ROOT
        print('Using HYPERVIEW2 from Drive:', HV2_ROOT)
    else:
        print('Copying HYPERVIEW2 to Drive. This can take a few minutes...')
        if DRIVE_HV2_ROOT.exists():
            shutil.rmtree(DRIVE_HV2_ROOT)
        shutil.copytree(HV2_ROOT, DRIVE_HV2_ROOT)
        HV2_ROOT = DRIVE_HV2_ROOT
        print('Cached HYPERVIEW2 on Drive:', HV2_ROOT)


## 3. Import kodu repo

In [ ]:
import csv
import json
import shutil
import time
from functools import partial
from typing import Any

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

from hsi_compression.downstream import (
    HYPERVIEW2_TARGET_COLUMNS,
    Hyperview2CompressionDataset,
    Hyperview2FeatureDataset,
    build_hyperview2_regressor,
    build_hyperview2_samples,
    collate_compression_batch,
    compute_regression_metrics,
    split_samples,
)
from hsi_compression.engine.checkpointing import load_checkpoint
from hsi_compression.metrics import (
    compute_compression_ratio_from_bpppc,
    masked_psnr,
    masked_sam_deg,
)
from hsi_compression.models.registry import build_model

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

## 4. Konfiguracja eksperymentu

Uzupelnij `CHECKPOINTS`, jezeli chcesz wygenerowac rekonstrukcje z checkpointow. Jezeli rekonstrukcje masz juz zapisane w canonical layoucie, uzupelnij `PRECOMPUTED_RECON_ROOTS`.

In [ ]:
MODALITY = 'prisma'
NORMALIZATION = 'none'
FEATURE_SET = 'mean_std_derivatives'
VAL_FRACTION = 0.2
SEED = 42
N_JOBS = -1

MODEL_NAMES = [
    'dummy_mean',
    'ridge',
    'pls',
    'knn',
    'extra_trees',
    'random_forest',
    'hist_gradient_boosting',
    # 'lightgbm',
    # 'catboost',
    # 'xgboost',
]

# Przyklad:
# CHECKPOINTS = [
#     {
#         'name': 'mamba_k4_hyperview2_ft',
#         'path': Path('/content/drive/MyDrive/hsi/checkpoints/mamba_k4_hyperview2_best.pt'),
#         'batch_size': 1,
#         'use_bitstream': True,
#     },
# ]
CHECKPOINT_DIR = DRIVE_CHECKPOINTS if 'DRIVE_CHECKPOINTS' in globals() else Path('/content/checkpoints')
CHECKPOINTS: list[dict[str, Any]] = [
    {
        'name': 'hyperview2_mamba_latent48',
        'path': CHECKPOINT_DIR / 'hyperview2_prisma_hierarchical_spectral_mamba_ae_latent48_best.pt',
        'batch_size': 1,
        'use_bitstream': True,
    },
]
# To compare more Mamba variants, uncomment or add entries below after the archive is extracted.
# CHECKPOINTS.extend([
#     {'name': 'hyperview2_mamba_rd_1e_4', 'path': CHECKPOINT_DIR / 'hyperview2_prisma_hierarchical_spectral_mamba_ae_latent48_rd_lambda_0_0001_best.pt', 'batch_size': 1, 'use_bitstream': True},
#     {'name': 'hyperview2_mamba_task_feature_rd', 'path': CHECKPOINT_DIR / 'hyperview2_prisma_hierarchical_spectral_mamba_ae_latent48_task_feature_rd_best.pt', 'batch_size': 1, 'use_bitstream': True},
# ])

# Przyklad: {'mamba_k4_hyperview2_ft': Path('/content/reconstructions/mamba_k4_hyperview2_ft/HYPERVIEW2')}
PRECOMPUTED_RECON_ROOTS: dict[str, Path] = {}

RECON_PARENT = DRIVE_RECONS / 'hyperview2' if 'DRIVE_RECONS' in globals() else Path('/content/reconstructions/hyperview2')
OUTPUT_DIR = (DRIVE_RESULTS / 'hyperview2_compression') if 'DRIVE_RESULTS' in globals() else Path('/content/artifacts/downstream/hyperview2_compression')
RECON_PARENT.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 5. Pomocnicze funkcje dla checkpointow i rekonstrukcji

In [ ]:
def safe_sample_stem(sample_id: str) -> str:
    return f'{int(sample_id):04d}' if str(sample_id).isdigit() else str(sample_id)


def prepare_recon_root(recon_root: Path, source_root: Path, modality: str) -> Path:
    recon_root.mkdir(parents=True, exist_ok=True)
    for filename in ['train_gt.csv', 'submission.csv', 'wavelengths.json']:
        src = source_root / filename
        if src.exists():
            shutil.copy2(src, recon_root / filename)
    for rel in ['train', 'test']:
        (recon_root / rel / 'hsi_satellite').mkdir(parents=True, exist_ok=True)
        (recon_root / rel / 'hsi_airborne').mkdir(parents=True, exist_ok=True)
        (recon_root / rel / 'msi_satellite').mkdir(parents=True, exist_ok=True)
    return recon_root


def call_model_forward(model, x, mask):
    try:
        return model(x, valid_mask=mask)
    except TypeError:
        return model(x)


def call_model_compress(model, x, mask):
    try:
        return model.compress(x, valid_mask=mask)
    except TypeError:
        return model.compress(x)


def call_model_decompress(model, packed):
    if 'latent' in packed:
        return model.decompress(latent=packed['latent'], z_shape=packed.get('z_shape'))
    kwargs = {'strings': packed['strings'], 'shape': packed['shape']}
    if packed.get('z_shape') is not None:
        kwargs['z_shape'] = packed['z_shape']
    return model.decompress(**kwargs)


def sum_string_bytes(obj) -> int:
    if isinstance(obj, bytes):
        return len(obj)
    if isinstance(obj, bytearray):
        return len(obj)
    if isinstance(obj, str):
        return len(obj.encode('utf-8'))
    if isinstance(obj, (list, tuple)):
        return sum(sum_string_bytes(item) for item in obj)
    raise TypeError(f'Unsupported strings container type: {type(obj)!r}')


def build_model_from_checkpoint(checkpoint_path: Path, in_channels: int, device: torch.device):
    raw = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
    cfg = raw.get('config', {})
    model_section = cfg.get('model', {})
    model_name = model_section.get('model_name')
    if not model_name:
        raise ValueError(f'Checkpoint {checkpoint_path} does not contain config.model.model_name')
    model_kwargs = dict(model_section.get('model_kwargs', {}))
    model_kwargs.pop('in_channels', None)
    model = build_model(model_name, in_channels=in_channels, **model_kwargs).to(device)
    load_checkpoint(checkpoint_path, model=model, optimizer=None, map_location=device)
    if hasattr(model, 'update'):
        model.update(force=True)
    model.eval()
    return model, cfg


def reconstruct_checkpoint(
    checkpoint: dict[str, Any],
    source_root: Path,
    recon_parent: Path,
    modality: str = 'prisma',
    normalization: str = 'none',
    split: str = 'train',
):
    name = checkpoint['name']
    checkpoint_path = Path(checkpoint['path'])
    if not checkpoint_path.exists():
        raise FileNotFoundError(checkpoint_path)
    batch_size = int(checkpoint.get('batch_size', 1))
    num_workers = int(checkpoint.get('num_workers', 2))
    pad_multiple = int(checkpoint.get('pad_multiple', 4))
    min_spatial_size = int(checkpoint.get('min_spatial_size', 4))
    use_bitstream = bool(checkpoint.get('use_bitstream', True))
    use_amp = bool(checkpoint.get('use_amp', DEVICE.type == 'cuda'))

    samples = build_hyperview2_samples(source_root, modality=modality, split=split)
    dataset = Hyperview2CompressionDataset(samples, modality=modality, normalization=normalization)
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        collate_fn=partial(collate_compression_batch, pad_multiple=pad_multiple, min_spatial_size=min_spatial_size),
    )
    first = dataset[0]
    in_channels = int(first['x'].shape[0])
    model, cfg = build_model_from_checkpoint(checkpoint_path, in_channels=in_channels, device=DEVICE)

    recon_root = prepare_recon_root(recon_parent / name / 'HYPERVIEW2', source_root, modality)
    out_dir = recon_root / split / 'hsi_satellite'

    totals = {
        'mse_sum': 0.0,
        'mae_sum': 0.0,
        'values': 0.0,
        'psnr_sum': 0.0,
        'sam_sum': 0.0,
        'samples': 0,
        'actual_bits': 0.0,
        'coded_values': 0.0,
        'encode_time_sec': 0.0,
        'decode_time_sec': 0.0,
    }

    with torch.no_grad():
        for batch in tqdm(loader, desc=f'reconstruct:{name}:{split}'):
            x = batch['x'].to(DEVICE, non_blocking=True)
            mask = batch['valid_mask'].to(DEVICE, non_blocking=True)
            start_encode = time.perf_counter()
            if use_bitstream and hasattr(model, 'compress') and hasattr(model, 'decompress'):
                packed = call_model_compress(model, x, mask)
                encode_time = time.perf_counter() - start_encode
                start_decode = time.perf_counter()
                decoded = call_model_decompress(model, packed)
                decode_time = time.perf_counter() - start_decode
                x_hat = decoded['x_hat'] if isinstance(decoded, dict) else decoded
                if isinstance(packed, dict) and packed.get('strings') is not None:
                    try:
                        totals['actual_bits'] += float(sum_string_bytes(packed['strings']) * 8)
                        totals['coded_values'] += float(sum(c * h * w for c, h, w in batch['original_shape']))
                    except Exception as exc:
                        print('actual_bpppc skipped:', exc)
            else:
                with torch.autocast(device_type=DEVICE.type, enabled=use_amp and DEVICE.type == 'cuda'):
                    outputs = call_model_forward(model, x, mask)
                encode_time = time.perf_counter() - start_encode
                decode_time = 0.0
                x_hat = outputs['x_hat'] if isinstance(outputs, dict) else outputs

            x_hat = x_hat.float().clamp(0.0, 1.0)
            totals['encode_time_sec'] += encode_time
            totals['decode_time_sec'] += decode_time

            totals['psnr_sum'] += float(masked_psnr(x_hat, x, mask).item()) * x.shape[0]
            totals['sam_sum'] += float(masked_sam_deg(x_hat, x, mask).item()) * x.shape[0]
            totals['samples'] += int(x.shape[0])
            mask_f = mask.float()
            totals['mse_sum'] += float(((x_hat - x) ** 2 * mask_f).sum().item())
            totals['mae_sum'] += float(((x_hat - x).abs() * mask_f).sum().item())
            totals['values'] += float(mask_f.sum().item())

            for idx, sample_id in enumerate(batch['sample_id']):
                c, h, w = batch['original_shape'][idx]
                arr = x_hat[idx, :c, :h, :w].detach().cpu().numpy().astype(np.float32)
                valid = mask[idx, :c, :h, :w].detach().cpu().numpy().astype(bool)
                np.savez_compressed(out_dir / f'{safe_sample_stem(sample_id)}.npz', data=arr, mask=valid)

    values = max(totals['values'], 1.0)
    actual_bpppc = None
    if totals['coded_values'] > 0:
        actual_bpppc = totals['actual_bits'] / totals['coded_values']
    summary = {
        'name': name,
        'checkpoint_path': str(checkpoint_path),
        'recon_root': str(recon_root),
        'samples': totals['samples'],
        'masked_mse': totals['mse_sum'] / values,
        'masked_mae': totals['mae_sum'] / values,
        'masked_psnr': totals['psnr_sum'] / max(totals['samples'], 1),
        'masked_sam_deg': totals['sam_sum'] / max(totals['samples'], 1),
        'actual_bpppc': actual_bpppc,
        'actual_cr_16bit': compute_compression_ratio_from_bpppc(actual_bpppc),
        'encode_time_sec': totals['encode_time_sec'],
        'decode_time_sec': totals['decode_time_sec'],
        'checkpoint_config': cfg,
    }
    (recon_root / 'reconstruction_summary.json').write_text(json.dumps(summary, indent=2, default=str), encoding='utf-8')
    return recon_root, summary

## 6. Generowanie rekonstrukcji

In [ ]:
generated_recon_roots: dict[str, Path] = {}
reconstruction_summaries = {}

for checkpoint in CHECKPOINTS:
    recon_root, summary = reconstruct_checkpoint(
        checkpoint,
        source_root=HV2_ROOT,
        recon_parent=RECON_PARENT,
        modality=MODALITY,
        normalization=NORMALIZATION,
        split='train',
    )
    generated_recon_roots[checkpoint['name']] = recon_root
    reconstruction_summaries[checkpoint['name']] = summary

RECON_ROOTS = {**PRECOMPUTED_RECON_ROOTS, **generated_recon_roots}
if not RECON_ROOTS:
    print('No reconstruction roots configured. Add CHECKPOINTS or PRECOMPUTED_RECON_ROOTS above.')
else:
    for name, root in RECON_ROOTS.items():
        print(name, '->', root)

## 7. Downstream feature matrices

In [ ]:
def make_feature_matrix(samples, modality: str, normalization: str, feature_set: str):
    dataset = Hyperview2FeatureDataset(samples, modality=modality, normalization=normalization, feature_set=feature_set)
    xs, ys, ids = [], [], []
    for idx in tqdm(range(len(dataset)), desc=f'features:{modality}:{feature_set}'):
        item = dataset[idx]
        xs.append(item['features'].numpy())
        ys.append(item['target'].numpy())
        ids.append(str(item['sample_id']))
    return np.stack(xs).astype(np.float32), np.stack(ys).astype(np.float32), ids


def samples_by_ids(root: Path, sample_ids: list[str], modality: str):
    samples = build_hyperview2_samples(root, modality=modality, split='train')
    by_id = {sample.sample_id: sample for sample in samples}
    missing = [sample_id for sample_id in sample_ids if sample_id not in by_id]
    if missing:
        raise KeyError(f'Missing reconstruction samples: {missing[:8]}')
    return [by_id[sample_id] for sample_id in sample_ids]


original_samples = build_hyperview2_samples(HV2_ROOT, modality=MODALITY, split='train')
train_samples, val_samples = split_samples(original_samples, val_fraction=VAL_FRACTION, seed=SEED)
x_train_orig, y_train, train_ids = make_feature_matrix(train_samples, MODALITY, NORMALIZATION, FEATURE_SET)
x_val_orig, y_val, val_ids = make_feature_matrix(val_samples, MODALITY, NORMALIZATION, FEATURE_SET)
baseline_mse = ((y_val - y_train.mean(axis=0, keepdims=True)) ** 2).mean(axis=0).astype(np.float32)

print('Original X train:', x_train_orig.shape, 'X val:', x_val_orig.shape)
print('Baseline MSE:', dict(zip(HYPERVIEW2_TARGET_COLUMNS, baseline_mse.tolist())))

## 8. Trenowanie i ewaluacja regresorow

In [ ]:
def run_regressors(x_train, y_train, x_val, y_val, baseline_mse, model_names, variant: str, mode: str):
    rows = []
    details = {}
    for name in model_names:
        start = time.perf_counter()
        row = {'variant': variant, 'mode': mode, 'model': name}
        try:
            regressor = build_hyperview2_regressor(
                name,
                random_state=SEED,
                n_jobs=N_JOBS,
                n_features=x_train.shape[1],
                n_samples=x_train.shape[0],
                n_targets=y_train.shape[1],
            )
            regressor.fit(x_train, y_train)
            fit_time = time.perf_counter() - start
            pred_start = time.perf_counter()
            y_pred = np.asarray(regressor.predict(x_val), dtype=np.float32)
            predict_time = time.perf_counter() - pred_start
            metrics = compute_regression_metrics(y_val, y_pred, baseline_mse)
            row.update({
                'status': 'ok',
                'hyperview_score': metrics['hyperview_score'],
                'mean_mse': metrics['mean_mse'],
                'mean_mae': metrics['mean_mae'],
                'fit_time_sec': fit_time,
                'predict_time_sec': predict_time,
            })
            for target, target_metrics in metrics['targets'].items():
                row[f'{target}_rmse'] = target_metrics['rmse']
                row[f'{target}_relative_mse'] = target_metrics['relative_mse']
            details[name] = {'status': 'ok', 'metrics': metrics, 'fit_time_sec': fit_time, 'predict_time_sec': predict_time}
        except Exception as exc:
            row.update({'status': 'failed', 'error': str(exc), 'fit_time_sec': time.perf_counter() - start})
            details[name] = {'status': 'failed', 'error': str(exc)}
        rows.append(row)
    return pd.DataFrame(rows), details


all_tables = []
all_details = {}

df, details = run_regressors(
    x_train_orig, y_train, x_val_orig, y_val, baseline_mse, MODEL_NAMES,
    variant='original', mode='original_train_to_original_val',
)
all_tables.append(df)
all_details['original/original_train_to_original_val'] = details

for variant_name, recon_root in RECON_ROOTS.items():
    recon_train_samples = samples_by_ids(Path(recon_root), train_ids, MODALITY)
    recon_val_samples = samples_by_ids(Path(recon_root), val_ids, MODALITY)
    x_train_recon, y_train_recon, _ = make_feature_matrix(recon_train_samples, MODALITY, NORMALIZATION, FEATURE_SET)
    x_val_recon, y_val_recon, _ = make_feature_matrix(recon_val_samples, MODALITY, NORMALIZATION, FEATURE_SET)

    df, details = run_regressors(
        x_train_orig, y_train, x_val_recon, y_val_recon, baseline_mse, MODEL_NAMES,
        variant=variant_name, mode='original_train_to_recon_val',
    )
    all_tables.append(df)
    all_details[f'{variant_name}/original_train_to_recon_val'] = details

    df, details = run_regressors(
        x_train_recon, y_train_recon, x_val_recon, y_val_recon, baseline_mse, MODEL_NAMES,
        variant=variant_name, mode='recon_train_to_recon_val',
    )
    all_tables.append(df)
    all_details[f'{variant_name}/recon_train_to_recon_val'] = details

results_df = pd.concat(all_tables, ignore_index=True)
results_df = results_df.sort_values(['variant', 'mode', 'status', 'hyperview_score'], na_position='last')
results_df

## 9. Zapis wynikow

In [ ]:
summary_csv = OUTPUT_DIR / 'compression_downstream_summary.csv'
metrics_json = OUTPUT_DIR / 'compression_downstream_metrics.json'

protocol = {
    'dataset': 'HYPERVIEW2',
    'dataset_root': str(HV2_ROOT),
    'source': 'EOTDL HYPERVIEW2 version 2 assets',
    'split_source': 'train_gt.csv fixed internal train/validation split',
    'modality': MODALITY,
    'normalization': NORMALIZATION,
    'feature_set': FEATURE_SET,
    'val_fraction': VAL_FRACTION,
    'seed': SEED,
    'target_columns': list(HYPERVIEW2_TARGET_COLUMNS),
    'train_samples': len(train_samples),
    'val_samples': len(val_samples),
    'train_sample_ids': train_ids,
    'val_sample_ids': val_ids,
    'model_names': MODEL_NAMES,
    'checkpoints': [{**cfg, 'path': str(cfg.get('path'))} for cfg in CHECKPOINTS],
    'recon_roots': {name: str(root) for name, root in RECON_ROOTS.items()},
}

payload = {
    'protocol': protocol,
    'baseline_mse': baseline_mse.tolist(),
    'reconstruction_summaries': reconstruction_summaries,
    'downstream_details': all_details,
    'rows': results_df.to_dict(orient='records'),
}

results_df.to_csv(summary_csv, index=False)
metrics_json.write_text(json.dumps(payload, indent=2, sort_keys=True, default=str), encoding='utf-8')

print('Saved:', summary_csv)
print('Saved:', metrics_json)
display(results_df)

In [ ]:
!find /content/artifacts/downstream/hyperview2_compression -maxdepth 2 -type f -print